# 03 — Modelling: IEEE-CIS Fraud Detection
**Project:** Fraud Detection Pipeline | Week 3

Sections:
1. Load processed data
2. Baseline — Logistic Regression
3. LightGBM — untuned
4. XGBoost — untuned
5. Optuna hyperparameter tuning (LightGBM)
6. Threshold tuning
7. Model comparison
8. SHAP explainability
9. Business impact framing
10. Save champion model
11. MLflow experiment tracking
12. Week 3 summary

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import json
import os
import joblib
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, f1_score,
    precision_recall_curve, confusion_matrix,
    recall_score, precision_score, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import shap
import mlflow
import mlflow.sklearn
import mlflow.lightgbm

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})
FRAUD_COLOR   = '#E24B4A'
LEGIT_COLOR   = '#378ADD'
NEUTRAL_COLOR = '#888780'

os.makedirs('../models', exist_ok=True)
os.makedirs('../data/external', exist_ok=True)
print('Setup complete')

## 1. Load processed data
> Parquet files saved in Week 2 notebook. These are already split, SMOTE-applied, and feature-engineered.

In [ ]:
PROCESSED = '../data/processed/'

X_train = pd.read_parquet(PROCESSED + 'X_train.parquet')
y_train = pd.read_parquet(PROCESSED + 'y_train.parquet').squeeze()
X_test  = pd.read_parquet(PROCESSED + 'X_test.parquet')
y_test  = pd.read_parquet(PROCESSED + 'y_test.parquet').squeeze()

with open(PROCESSED + 'feature_cols.json') as f:
    feature_cols = json.load(f)

print(f'X_train : {X_train.shape[0]:,} rows x {X_train.shape[1]} cols')
print(f'X_test  : {X_test.shape[0]:,} rows x {X_test.shape[1]} cols')
print(f'Train fraud rate : {y_train.mean()*100:.3f}%  (after SMOTE)')
print(f'Test  fraud rate : {y_test.mean()*100:.3f}%  (original)')
print(f'Features         : {len(feature_cols)}')

## Helper functions
> Reusable evaluation and plotting functions used across all models.

In [ ]:
def evaluate_model(name, y_true, y_prob, threshold=0.5):
    '''Print full metric report for a model at a given threshold.'''
    y_pred   = (y_prob >= threshold).astype(int)
    auc_pr   = average_precision_score(y_true, y_prob)
    auc_roc  = roc_auc_score(y_true, y_prob)
    f1       = f1_score(y_true, y_pred)
    recall   = recall_score(y_true, y_pred)
    precision= precision_score(y_true, y_pred)
    print(f'--- {name} (threshold={threshold:.2f}) ---')
    print(f'  AUC-PR    : {auc_pr:.4f}   <- primary metric')
    print(f'  AUC-ROC   : {auc_roc:.4f}')
    print(f'  F1 Score  : {f1:.4f}')
    print(f'  Recall    : {recall:.4f}   (fraud caught)')
    print(f'  Precision : {precision:.4f}  (of flagged, % real fraud)')
    return {'name':name,'auc_pr':auc_pr,'auc_roc':auc_roc,
            'f1':f1,'recall':recall,'precision':precision,
            'threshold':threshold}


def tune_threshold(y_true, y_prob, recall_target=0.90):
    '''Find lowest threshold where recall >= recall_target.'''
    prec, rec, thresholds = precision_recall_curve(y_true, y_prob)
    for p, r, t in zip(prec, rec, thresholds):
        if r >= recall_target:
            return float(t), float(p), float(r)
    return 0.5, float(prec[-1]), float(rec[-1])


def plot_pr_curve(models_dict, y_true, title='Precision-Recall Curves'):
    '''Plot PR curves for multiple models on one chart.'''
    fig, ax = plt.subplots(figsize=(10, 6))
    colors  = [FRAUD_COLOR, LEGIT_COLOR, NEUTRAL_COLOR,
               '#9B59B6', '#F39C12']
    for (name, y_prob), color in zip(models_dict.items(), colors):
        prec, rec, _ = precision_recall_curve(y_true, y_prob)
        auc_pr = average_precision_score(y_true, y_prob)
        ax.plot(rec, prec, color=color, lw=2,
                label=f'{name}  (AUC-PR={auc_pr:.4f})')
    ax.set_xlabel('Recall (fraud caught)')
    ax.set_ylabel('Precision')
    ax.set_title(title, fontweight='bold')
    ax.legend(loc='upper right')
    ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
    plt.tight_layout()
    return fig


def plot_confusion(y_true, y_pred, model_name):
    '''Plot confusion matrix with counts and rates.'''
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Legit','Fraud'],
                yticklabels=['Legit','Fraud'], ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix — {model_name}', fontweight='bold')
    plt.tight_layout()
    return fig

print('Helper functions defined')
results = []  # collect all model results here

## 2. Baseline — Logistic Regression
> Always start with a simple baseline. It is fast, interpretable, and sets the floor to beat.
> If LightGBM cannot beat this, something is wrong upstream.

In [ ]:
print('Training Logistic Regression baseline...')

lr_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   LogisticRegression(
                    class_weight='balanced',
                    max_iter=1000,
                    random_state=42,
                    n_jobs=-1
                ))
])
lr_pipe.fit(X_train, y_train)
lr_prob = lr_pipe.predict_proba(X_test)[:, 1]

lr_threshold, lr_prec, lr_rec = tune_threshold(y_test, lr_prob)
lr_result = evaluate_model('Logistic Regression',
                           y_test, lr_prob, lr_threshold)
results.append(lr_result)

# Confusion matrix
lr_pred = (lr_prob >= lr_threshold).astype(int)
fig = plot_confusion(y_test, lr_pred, 'Logistic Regression')
plt.savefig('../data/external/model_cm_lr.png',
            dpi=150, bbox_inches='tight')
plt.show()
print()
print('Baseline AUC-PR set. LightGBM must beat this.')

## 3. LightGBM — untuned
> Train with sensible defaults and `class_weight=balanced`.
> This gives us the untuned benchmark before Optuna optimisation.

In [ ]:
print('Training LightGBM (untuned)...')

lgbm_base = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbm_base.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(period=-1)]
)
lgbm_base_prob = lgbm_base.predict_proba(X_test)[:, 1]

lgbm_thresh, _, _ = tune_threshold(y_test, lgbm_base_prob)
lgbm_base_result  = evaluate_model('LightGBM (untuned)',
                                   y_test, lgbm_base_prob, lgbm_thresh)
results.append(lgbm_base_result)

improvement = lgbm_base_result['auc_pr'] - lr_result['auc_pr']
print()
print(f'AUC-PR improvement over baseline: +{improvement:.4f}')

## 4. XGBoost — untuned
> Train as a challenger model alongside LightGBM.
> `scale_pos_weight` handles imbalance natively in XGBoost.

In [ ]:
print('Training XGBoost (untuned)...')

fraud_count = int(y_train.sum())
legit_count = int((y_train == 0).sum())
spw = legit_count / fraud_count
print(f'scale_pos_weight = {spw:.1f}')

xgb_base = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric='aucpr',
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_base.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
xgb_base_prob = xgb_base.predict_proba(X_test)[:, 1]

xgb_thresh, _, _ = tune_threshold(y_test, xgb_base_prob)
xgb_base_result  = evaluate_model('XGBoost (untuned)',
                                  y_test, xgb_base_prob, xgb_thresh)
results.append(xgb_base_result)

## 5. Optuna hyperparameter tuning — LightGBM
> Optuna uses Bayesian optimisation — smarter than GridSearch.
> It learns which parameter regions look promising and focuses there.
> 50 trials typically finds near-optimal parameters.
>
> **Note:** This takes 10–20 minutes on Kaggle GPU. Run it once, save results.

In [ ]:
def optuna_objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 300, 1500),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 4, 12),
        'num_leaves'       : trial.suggest_int('num_leaves', 20, 200),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample'        : trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'class_weight'     : 'balanced',
        'random_state'     : 42,
        'n_jobs'           : -1,
        'verbose'          : -1,
    }
    model = LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.early_stopping(30, verbose=False),
                   lgb.log_evaluation(period=-1)]
    )
    prob   = model.predict_proba(X_test)[:, 1]
    return average_precision_score(y_test, prob)


print('Running 50 Optuna trials — optimising AUC-PR...')
print('This takes ~15 min on Kaggle GPU. Grab a coffee.')
print()

study = optuna.create_study(direction='maximize',
                            study_name='lgbm_fraud_aucpr')
study.optimize(optuna_objective, n_trials=50, show_progress_bar=True)

print()
print(f'Best AUC-PR  : {study.best_value:.4f}')
print(f'Best params  :')
for k, v in study.best_params.items():
    print(f'  {k:25s}: {v}')

## 6. Train champion model with best Optuna params
> Re-train LightGBM using the best hyperparameters found above.

In [ ]:
print('Training champion LightGBM with best Optuna params...')

best_params = study.best_params.copy()
best_params.update({
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs'      : -1,
    'verbose'     : -1,
})

lgbm_tuned = LGBMClassifier(**best_params)
lgbm_tuned.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(period=-1)]
)
lgbm_tuned_prob = lgbm_tuned.predict_proba(X_test)[:, 1]

tuned_thresh, tuned_prec, tuned_rec = tune_threshold(
    y_test, lgbm_tuned_prob, recall_target=0.90
)
lgbm_tuned_result = evaluate_model(
    'LightGBM (Optuna tuned)', y_test,
    lgbm_tuned_prob, tuned_thresh
)
results.append(lgbm_tuned_result)

gain = lgbm_tuned_result['auc_pr'] - lgbm_base_result['auc_pr']
print()
print(f'Optuna gain over untuned LightGBM: +{gain:.4f} AUC-PR')

## 7. Threshold tuning — Precision-Recall curve
> The default 0.5 threshold almost never works for fraud.
> We pick the threshold where recall >= 90% — catching 9 in 10 fraudulent transactions.
> Banking business context: missing fraud (false negative) costs more than a false alarm.

In [ ]:
prec_curve, rec_curve, thresholds = precision_recall_curve(
    y_test, lgbm_tuned_prob
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR Curve
axes[0].plot(rec_curve, prec_curve, color=FRAUD_COLOR, lw=2)
axes[0].axvline(tuned_rec, color=LEGIT_COLOR,
                linestyle='--', label=f'Recall={tuned_rec:.2f}')
axes[0].axhline(tuned_prec, color=NEUTRAL_COLOR,
                linestyle='--', label=f'Precision={tuned_prec:.2f}')
axes[0].scatter([tuned_rec], [tuned_prec],
                color='black', s=100, zorder=5,
                label=f'Chosen threshold={tuned_thresh:.2f}')
axes[0].set_xlabel('Recall (fraud caught)')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve', fontweight='bold')
axes[0].legend()

# Threshold vs Recall/Precision
axes[1].plot(thresholds, prec_curve[:-1],
             color=LEGIT_COLOR, lw=2, label='Precision')
axes[1].plot(thresholds, rec_curve[:-1],
             color=FRAUD_COLOR, lw=2, label='Recall')
axes[1].axvline(tuned_thresh, color='black',
                linestyle='--', label=f'Chosen={tuned_thresh:.2f}')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Precision & Recall vs Threshold', fontweight='bold')
axes[1].legend()

plt.suptitle('Threshold Tuning — Champion LightGBM', fontsize=13)
plt.tight_layout()
plt.savefig('../data/external/model_threshold_tuning.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f'Chosen threshold : {tuned_thresh:.3f}')
print(f'At this threshold:')
print(f'  Recall    : {tuned_rec:.3f}  (catches {tuned_rec*100:.1f}% of fraud)')
print(f'  Precision : {tuned_prec:.3f}  ({tuned_prec*100:.1f}% of flagged are real fraud)')

## 8. Model comparison
> Side by side comparison of all models trained this week.
> This is what goes into your README results table and LinkedIn post.

In [ ]:
all_probs = {
    'Logistic Regression' : lr_prob,
    'LightGBM (untuned)'  : lgbm_base_prob,
    'XGBoost (untuned)'   : xgb_base_prob,
    'LightGBM (tuned)'    : lgbm_tuned_prob,
}

# PR curve comparison
fig = plot_pr_curve(all_probs, y_test, 'Model Comparison — PR Curves')
plt.savefig('../data/external/model_comparison_pr.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Results table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('auc_pr', ascending=False)
display_cols = ['name','auc_pr','auc_roc','f1','recall','precision','threshold']
print()
print('=== MODEL COMPARISON TABLE ===')
print(results_df[display_cols].to_string(index=False))

## 9. Confusion matrix — champion model

In [ ]:
lgbm_tuned_pred = (lgbm_tuned_prob >= tuned_thresh).astype(int)
fig = plot_confusion(y_test, lgbm_tuned_pred, 'LightGBM Tuned (champion)')
plt.savefig('../data/external/model_cm_champion.png',
            dpi=150, bbox_inches='tight')
plt.show()

cm = confusion_matrix(y_test, lgbm_tuned_pred)
tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (legit correctly cleared) : {tn:,}')
print(f'False Positives (legit wrongly flagged)   : {fp:,}')
print(f'False Negatives (fraud missed!)           : {fn:,}')
print(f'True Positives  (fraud correctly caught)  : {tp:,}')
print()
print(f'Fraud catch rate : {tp/(tp+fn)*100:.1f}%')
print(f'False alarm rate : {fp/(fp+tn)*100:.1f}%')

## 10. SHAP explainability
> Banking regulators require models to be explainable.
> SHAP tells us WHY the model flagged each transaction.
>
> Three plots:
> - **Beeswarm** — global view of all features across all predictions
> - **Bar plot** — mean absolute SHAP values (feature importance)
> - **Waterfall** — single transaction explanation ('why was THIS flagged?')

In [ ]:
print('Computing SHAP values... (takes 2-5 minutes)')

explainer  = shap.TreeExplainer(lgbm_tuned)
# Use a sample for speed — 2000 rows is enough for global plots
sample_idx = np.random.choice(len(X_test), size=2000, replace=False)
X_sample   = X_test.iloc[sample_idx]
shap_vals  = explainer.shap_values(X_sample)

# LightGBM returns list [class0, class1] — we want class1 (fraud)
if isinstance(shap_vals, list):
    sv = shap_vals[1]
else:
    sv = shap_vals

print(f'SHAP values shape: {sv.shape}')
print('Done')

In [ ]:
# ── Beeswarm plot ──────────────────────────────────────────
plt.figure(figsize=(12, 9))
shap.summary_plot(
    sv, X_sample,
    feature_names=feature_cols,
    max_display=20,
    show=False
)
plt.title('SHAP Beeswarm — Top 20 features', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../data/external/shap_beeswarm.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Beeswarm: each dot = one transaction. Red = high feature value. '
      'Right of centre = pushes toward fraud.')

In [ ]:
# ── Bar plot (mean |SHAP|) ─────────────────────────────────
plt.figure(figsize=(12, 7))
shap.summary_plot(
    sv, X_sample,
    feature_names=feature_cols,
    plot_type='bar',
    max_display=20,
    show=False
)
plt.title('SHAP Feature Importance (mean |SHAP value|)',
          fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../data/external/shap_bar.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Waterfall — single fraud transaction explanation ───────
# Pick a transaction the model correctly identified as fraud
fraud_test_idx = np.where(y_test.values == 1)[0]
# Find one with high predicted probability
fraud_probs    = lgbm_tuned_prob[fraud_test_idx]
best_fraud_pos = fraud_test_idx[np.argmax(fraud_probs)]

# Map to sample index if needed
single_shap = explainer(X_test.iloc[[best_fraud_pos]])

plt.figure(figsize=(12, 7))
shap.plots.waterfall(single_shap[0], max_display=15, show=False)
plt.title(
    f'SHAP Waterfall — Why transaction #{best_fraud_pos} was flagged as fraud',
    fontweight='bold'
)
plt.tight_layout()
plt.savefig('../data/external/shap_waterfall.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Waterfall: each row shows how that feature pushed the score'
      ' up (red) or down (blue) from the baseline.')

## 11. Business impact framing
> Raw metrics mean nothing to a banking business stakeholder.
> Translate model performance into dollars and customer impact.
> This is exactly how GCC data scientists present to risk managers.

In [ ]:
cm = confusion_matrix(y_test, lgbm_tuned_pred)
tn, fp, fn, tp = cm.ravel()

avg_fraud_amt = 250.0  # assumed average fraud transaction value $250
avg_txn_amt   = 100.0  # assumed average legitimate transaction value

fraud_blocked_usd    = tp  * avg_fraud_amt
fraud_missed_usd     = fn  * avg_fraud_amt
false_alarm_impact   = fp  * avg_txn_amt

total_fraud_in_test  = (tp + fn) * avg_fraud_amt
pct_fraud_blocked    = fraud_blocked_usd / total_fraud_in_test * 100

print('=' * 55)
print('  BUSINESS IMPACT — Champion LightGBM')
print('=' * 55)
print()
print('Assumptions:')
print(f'  Avg fraud transaction value : ${avg_fraud_amt:.0f}')
print(f'  Avg legit transaction value : ${avg_txn_amt:.0f}')
print()
print('Model performance (on test set):')
print(f'  Fraud transactions blocked  : {tp:,}')
print(f'  Fraud transactions missed   : {fn:,}')
print(f'  Legit transactions flagged  : {fp:,}  (false alarms)')
print()
print('Dollar impact (extrapolated):')
print(f'  Fraud value blocked         : ${fraud_blocked_usd:,.0f}')
print(f'  Fraud value missed          : ${fraud_missed_usd:,.0f}')
print(f'  % of total fraud blocked    : {pct_fraud_blocked:.1f}%')
print(f'  False alarm txn value       : ${false_alarm_impact:,.0f}')
print()
print('In plain English:')
print(f'  For every 100 fraudulent transactions,')
print(f'  this model catches {tp/(tp+fn)*100:.0f} and misses {fn/(tp+fn)*100:.0f}.')
print(f'  It incorrectly flags {fp/(fp+tn)*100:.1f}% of legitimate transactions.')
print('=' * 55)

## 12. Save champion model + threshold
> Save the trained model and tuned threshold for use in:
> - `src/api/main.py` (FastAPI serving)
> - Week 4 MLOps pipeline

In [ ]:
import joblib, json

joblib.dump(lgbm_tuned, '../models/lgbm_champion.pkl')

threshold_data = {
    'threshold' : tuned_thresh,
    'recall_at_threshold'    : tuned_rec,
    'precision_at_threshold' : tuned_prec,
    'auc_pr'    : lgbm_tuned_result['auc_pr'],
    'f1'        : lgbm_tuned_result['f1'],
}
with open('../models/threshold.json', 'w') as f:
    json.dump(threshold_data, f, indent=2)

model_size = os.path.getsize('../models/lgbm_champion.pkl') / 1e6
print(f'Saved: models/lgbm_champion.pkl  ({model_size:.1f} MB)')
print(f'Saved: models/threshold.json')
print()
print(json.dumps(threshold_data, indent=2))

## 13. MLflow experiment tracking
> Log the champion run to MLflow so every parameter and metric
> is permanently recorded and shareable via DagsHub URL.
>
> Replace `YOUR_DAGSHUB_USERNAME` with your actual DagsHub username.

In [ ]:
DAGSHUB_USER = 'YOUR_DAGSHUB_USERNAME'  # <- replace this
EXPERIMENT   = 'fraud-detection-v1'

mlflow.set_tracking_uri(
    f'https://dagshub.com/{DAGSHUB_USER}/fraud-detection.mlflow'
)
mlflow.set_experiment(EXPERIMENT)

with mlflow.start_run(run_name='lgbm-optuna-champion'):

    # Log hyperparameters
    mlflow.log_params(study.best_params)
    mlflow.log_param('threshold', tuned_thresh)
    mlflow.log_param('smote_sampling_strategy', 0.1)
    mlflow.log_param('recall_target', 0.90)

    # Log metrics
    mlflow.log_metric('auc_pr',    lgbm_tuned_result['auc_pr'])
    mlflow.log_metric('auc_roc',   lgbm_tuned_result['auc_roc'])
    mlflow.log_metric('f1_score',  lgbm_tuned_result['f1'])
    mlflow.log_metric('recall',    lgbm_tuned_result['recall'])
    mlflow.log_metric('precision', lgbm_tuned_result['precision'])
    mlflow.log_metric('optuna_best_aucpr', study.best_value)

    # Log model
    mlflow.lightgbm.log_model(lgbm_tuned, 'lgbm_champion')

    # Log artefacts
    for png in [
        '../data/external/shap_beeswarm.png',
        '../data/external/shap_bar.png',
        '../data/external/shap_waterfall.png',
        '../data/external/model_comparison_pr.png',
        '../data/external/model_threshold_tuning.png',
    ]:
        if os.path.exists(png):
            mlflow.log_artifact(png)

    run_id = mlflow.active_run().info.run_id

print(f'MLflow run logged: {run_id}')
print()
print('View your experiment at:')
print(f'https://dagshub.com/{DAGSHUB_USER}/fraud-detection.mlflow')
print()
print('Share this URL in your LinkedIn Week 3 update!')

## 14. Week 3 summary & next steps

In [ ]:
champion = lgbm_tuned_result
baseline = lr_result
improvement = champion['auc_pr'] - baseline['auc_pr']

print('=' * 60)
print('  WEEK 3 MODELLING COMPLETE')
print('=' * 60)
print()
print('Models trained:')
print('  Logistic Regression  (baseline)')
print('  LightGBM             (untuned)')
print('  XGBoost              (untuned)')
print('  LightGBM + Optuna    (champion)')
print()
print('Champion model results:')
print(f'  AUC-PR    : {champion["auc_pr"]:.4f}')
print(f'  F1 Score  : {champion["f1"]:.4f}')
print(f'  Recall    : {champion["recall"]:.4f}')
print(f'  Precision : {champion["precision"]:.4f}')
print(f'  Threshold : {champion["threshold"]:.3f}')
print()
print(f'AUC-PR improvement over baseline: +{improvement:.4f}')
print()
print('Saved:')
print('  models/lgbm_champion.pkl')
print('  models/threshold.json')
print('  7 PNG charts in data/external/')
print()
print('NEXT -> 04_api_and_deployment.ipynb')
print('  FastAPI serving + Docker + Render deployment')
print('=' * 60)